<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/day9_vulnerability_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup
!pip install osmnx geopandas folium pandas numpy matplotlib -q

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import matplotlib.pyplot as plt

print('✅ Day 9 ready!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 7.7 MB/s eta 0:00:00
✅ Day 9 ready!


In [ ]:
# Cell 2 — School data
import pandas as pd
import numpy as np

schools = {
    'school_name': [
        'School Puducherry Border',
        'Panchayat School Nagapattinam',
        'School Ramanathapuram',
        'Govt School Cuddalore',
        'School Tuticorin',
        'School Kanchipuram',
        'Govt School Tiruchirappalli',
        'Panchayat School Tirunelveli',
        'Govt School Thanjavur',
        'High School Vellore',
        'School Villupuram',
        'Govt School Madurai',
        'Govt High School Chennai',
        'High School Coimbatore',
        'Govt School Salem'
    ],
    'latitude': [
        11.93, 10.76, 9.37, 11.75, 8.80,
        12.83, 10.79, 8.71, 10.78, 12.92,
        11.93, 9.93, 13.08, 11.01, 11.65
    ],
    'longitude': [
        79.83, 79.84, 78.83, 79.75, 78.15,
        79.70, 78.68, 77.69, 79.13, 79.13,
        79.49, 78.12, 80.27, 76.96, 78.16
    ],
    'connectivity': [
        'connected', 'none', 'none', 'none',
        'connected', 'none', 'none', 'none',
        'connected', 'connected', 'none',
        'connected', 'connected', 'connected', 'connected'
    ],
    'flood_score': [
        96.7, 56.8, 43.5, 39.8, 39.1,
        33.3, 31.4, 30.7, 26.5, 22.1,
        22.9, 13.0, 13.8, 4.8, 4.9
    ],
    'cyclone_norm': [
        87.5, 79.2, 45.8, 87.5, 37.5,
        79.2, 50.0, 37.5, 58.3, 66.7,
        83.3, 41.7, 100.0, 25.0, 62.5
    ]
}

df = pd.DataFrame(schools)
print(f'✅ {len(df)} schools loaded!')
print(df[['school_name', 'connectivity']].to_string(index=False))

✅ 15 schools loaded!
                  school_name connectivity
     School Puducherry Border    connected
Panchayat School Nagapattinam         none
        School Ramanathapuram         none
        Govt School Cuddalore         none
             School Tuticorin    connected
           School Kanchipuram         none
  Govt School Tiruchirappalli         none
 Panchayat School Tirunelveli         none
        Govt School Thanjavur    connected
          High School Vellore    connected
            School Villupuram         none
          Govt School Madurai    connected
     Govt High School Chennai    connected
       High School Coimbatore    connected
            Govt School Salem    connected


In [ ]:
# Search city by city instead
import osmnx as ox
import pandas as pd
import numpy as np
from shapely.geometry import Point
import geopandas as gpd

print('Searching hospitals city by city...')

# Key Tamil Nadu cities
cities = [
    'Chennai', 'Cuddalore', 'Nagapattinam',
    'Thanjavur', 'Madurai', 'Coimbatore',
    'Salem', 'Vellore', 'Tiruchirappalli'
]

all_hospitals = []

for city in cities:
    try:
        h = ox.features_from_place(
            city + ', Tamil Nadu, India',
            tags={'amenity': 'hospital'}
        )
        if len(h) > 0:
            # Get centroid for each hospital
            h['city'] = city
            all_hospitals.append(h[['geometry', 'city']])
            print(f'✅ {city}: {len(h)} hospitals')
    except Exception as e:
        print(f'⚠ {city}: skipped')

if all_hospitals:
    hospitals_gdf = pd.concat(all_hospitals)
    hospitals_gdf = hospitals_gdf.to_crs('EPSG:4326')
    # Get centroids
    hospitals_gdf['lat'] = hospitals_gdf.geometry.centroid.y
    hospitals_gdf['lon'] = hospitals_gdf.geometry.centroid.x
    print(f'\n✅ Total hospitals found: {len(hospitals_gdf)}')
else:
    print('No hospitals found — using manual data')

Searching hospitals city by city...
✅ Chennai: 633 hospitals
✅ Cuddalore: 90 hospitals
✅ Nagapattinam: 30 hospitals
✅ Thanjavur: 269 hospitals
✅ Madurai: 215 hospitals
✅ Coimbatore: 378 hospitals
✅ Salem: 364 hospitals
✅ Vellore: 79 hospitals
✅ Tiruchirappalli: 110 hospitals

✅ Total hospitals found: 2168


/tmp/ipykernel_4680/1904741035.py:37: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals_gdf['lat'] = hospitals_gdf.geometry.centroid.y
/tmp/ipykernel_4680/1904741035.py:38: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals_gdf['lon'] = hospitals_gdf.geometry.centroid.x


In [ ]:
# Distance to nearest hospital
import numpy as np
import pandas as pd

# Haversine distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Hospital coordinates
hosp_lats = hospitals_gdf['lat'].values
hosp_lons = hospitals_gdf['lon'].values

print('Calculating distances to nearest hospital...')

dist_results = []
for _, row in df.iterrows():
    distances = haversine(
        row['latitude'], row['longitude'],
        hosp_lats, hosp_lons
    )
    min_dist = np.min(distances)
    dist_results.append({
        'school_name': row['school_name'],
        'nearest_hospital_km': round(min_dist, 2)
    })

dist_df = pd.DataFrame(dist_results)
dist_df = dist_df.sort_values(
    'nearest_hospital_km', ascending=False
).reset_index(drop=True)

print('\n=== DISTANCE TO NEAREST HOSPITAL ===\n')
for _, row in dist_df.iterrows():
    bar = '█' * int(row['nearest_hospital_km'])
    print(f"{row['school_name'][:35]:<35} | "
          f"{row['nearest_hospital_km']:5.1f} km {bar}")

Calculating distances to nearest hospital...

=== DISTANCE TO NEAREST HOSPITAL ===

Panchayat School Tirunelveli        | 136.1 km ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
School Tuticorin                    | 119.3 km ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
School Ramanathapuram               |  94.1 km ██████████████████████████████████████████████████████████████████████████████████████████████
School Kanchipuram                  |  51.5 km ███████████████████████████████████████████████████
School Puducherry Border            |  19.1 km ███████████████████
School Villupuram                   |  18.8 km ██████████████████
Govt School Tiruchirappalli         |   0.7 km 
High School Vellore                 |   0.6 km 
Govt School Madurai                 |   0.6 km 
Govt School Salem                   |   0.5 km 

In [2]:
# Rebuild df + vulnerability index
import pandas as pd
import numpy as np

# Rebuild school dataframe
schools = {
    'school_name': [
        'School Puducherry Border',
        'Panchayat School Nagapattinam',
        'School Ramanathapuram',
        'Govt School Cuddalore',
        'School Tuticorin',
        'School Kanchipuram',
        'Govt School Tiruchirappalli',
        'Panchayat School Tirunelveli',
        'Govt School Thanjavur',
        'High School Vellore',
        'School Villupuram',
        'Govt School Madurai',
        'Govt High School Chennai',
        'High School Coimbatore',
        'Govt School Salem'
    ],
    'latitude': [
        11.93, 10.76, 9.37, 11.75, 8.80,
        12.83, 10.79, 8.71, 10.78, 12.92,
        11.93, 9.93, 13.08, 11.01, 11.65
    ],
    'longitude': [
        79.83, 79.84, 78.83, 79.75, 78.15,
        79.70, 78.68, 77.69, 79.13, 79.13,
        79.49, 78.12, 80.27, 76.96, 78.16
    ],
    'connectivity': [
        'connected', 'none', 'none', 'none',
        'connected', 'none', 'none', 'none',
        'connected', 'connected', 'none',
        'connected', 'connected', 'connected', 'connected'
    ],
    'flood_score': [
        96.7, 56.8, 43.5, 39.8, 39.1,
        33.3, 31.4, 30.7, 26.5, 22.1,
        22.9, 13.0, 13.8, 4.8, 4.9
    ],
    'cyclone_norm': [
        87.5, 79.2, 45.8, 87.5, 37.5,
        79.2, 50.0, 37.5, 58.3, 66.7,
        83.3, 41.7, 100.0, 25.0, 62.5
    ],
    'nearest_hospital_km': [
        19.1, 0.1, 94.1, 0.2, 119.3,
        51.5, 0.7, 136.1, 0.2, 0.6,
        18.8, 0.6, 0.3, 0.2, 0.5
    ]
}

df = pd.DataFrame(schools)

# Urban classification
urban_schools = [
    'Govt High School Chennai',
    'High School Coimbatore',
    'Govt School Madurai',
    'Govt School Salem',
    'High School Vellore',
    'Govt School Tiruchirappalli'
]

df['is_urban'] = df['school_name'].apply(
    lambda x: 1 if x in urban_schools else 0
)
df['is_rural'] = 1 - df['is_urban']

# Vulnerability components
df['conn_vuln'] = df['connectivity'].apply(
    lambda x: 100 if x == 'none' else 0
)

max_dist = df['nearest_hospital_km'].max()
df['hospital_vuln'] = (
    df['nearest_hospital_km'] / max_dist * 100
).round(1)

df['rural_vuln'] = df['is_rural'] * 60

# Combined vulnerability index
df['vulnerability_index'] = (
    0.40 * df['conn_vuln'] +
    0.35 * df['hospital_vuln'] +
    0.25 * df['rural_vuln']
).round(1)

# Vulnerability tier
def vuln_tier(score):
    if score >= 60:   return 'HIGH'
    elif score >= 30: return 'MEDIUM'
    else:             return 'LOW'

df['vuln_tier'] = df['vulnerability_index'].apply(vuln_tier)
df_vuln = df.sort_values(
    'vulnerability_index', ascending=False
).reset_index(drop=True)

print('=== VULNERABILITY INDEX ===')
print('Connectivity(40%) + Hospital(35%) + Rural(25%)\n')
print(f"{'School':<35} {'Conn':>5} {'Hosp':>6} {'Rural':>6} {'Index':>7} {'Tier'}")
print('-' * 72)
for _, row in df_vuln.iterrows():
    print(
        f"{row['school_name'][:34]:<35} "
        f"{row['conn_vuln']:>5.0f} "
        f"{row['hospital_vuln']:>6.1f} "
        f"{row['rural_vuln']:>6.0f} "
        f"{row['vulnerability_index']:>6.1f} "
        f"{row['vuln_tier']}"
    )

=== VULNERABILITY INDEX ===
Connectivity(40%) + Hospital(35%) + Rural(25%)

School                               Conn   Hosp  Rural   Index Tier
------------------------------------------------------------------------
Panchayat School Tirunelveli          100  100.0     60   90.0 HIGH
School Ramanathapuram                 100   69.1     60   79.2 HIGH
School Kanchipuram                    100   37.8     60   68.2 HIGH
School Villupuram                     100   13.8     60   59.8 MEDIUM
Panchayat School Nagapattinam         100    0.1     60   55.0 MEDIUM
Govt School Cuddalore                 100    0.1     60   55.0 MEDIUM
School Tuticorin                        0   87.7     60   45.7 MEDIUM
Govt School Tiruchirappalli           100    0.5      0   40.2 MEDIUM
School Puducherry Border                0   14.0     60   19.9 LOW
Govt School Thanjavur                   0    0.1     60   15.0 LOW
High School Vellore                     0    0.4      0    0.1 LOW
Govt School Madurai        

In [3]:
# Cell 6 — Full IPCC H x E x V Risk Framework
import pandas as pd
import numpy as np

# Hazard score (flood + cyclone combined)
df['hazard_score'] = (
    0.60 * df['flood_score'] +
    0.40 * df['cyclone_norm']
).round(1)

# Exposure score
# Schools in coastal districts = high exposure
coastal_schools = [
    'School Puducherry Border',
    'Panchayat School Nagapattinam',
    'School Ramanathapuram',
    'Govt School Cuddalore',
    'School Tuticorin',
    'School Kanchipuram',
    'School Villupuram',
    'Govt High School Chennai',
    'Panchayat School Tirunelveli'
]

df['exposure_score'] = df['school_name'].apply(
    lambda x: 85 if x in coastal_schools else 45
)

# Normalise all components 0-1
df['H'] = df['hazard_score'] / 100
df['E'] = df['exposure_score'] / 100
df['V'] = df['vulnerability_index'] / 100

# IPCC Risk = H x E x V (multiplicative)
# Scale to 0-100
df['hev_score'] = (df['H'] * df['E'] * df['V'] * 100).round(1)

# Also additive version for comparison
df['hev_additive'] = (
    0.40 * df['hazard_score'] +
    0.30 * df['exposure_score'] +
    0.30 * df['vulnerability_index']
).round(1)

# Final HEV risk tier
def hev_risk(score):
    if score >= 40:   return 'CRITICAL'
    elif score >= 20: return 'HIGH'
    elif score >= 10: return 'MEDIUM'
    else:             return 'LOW'

df['hev_risk'] = df['hev_additive'].apply(hev_risk)
df_hev = df.sort_values(
    'hev_additive', ascending=False
).reset_index(drop=True)
df_hev['rank'] = df_hev.index + 1

print('=== FULL H x E x V RISK FRAMEWORK ===')
print('Hazard(40%) x Exposure(30%) x Vulnerability(30%)\n')
print(f"{'#':<3} {'School':<32} {'H':>6} {'E':>6} {'V':>6} {'Score':>7} {'Risk'}")
print('-' * 70)
for _, row in df_hev.iterrows():
    print(
        f"#{int(row['rank']):<2} "
        f"{row['school_name'][:31]:<32} "
        f"{row['hazard_score']:>5.1f} "
        f"{row['exposure_score']:>5.0f} "
        f"{row['vulnerability_index']:>5.1f} "
        f"{row['hev_additive']:>6.1f} "
        f"{row['hev_risk']}"
    )

=== FULL H x E x V RISK FRAMEWORK ===
Hazard(40%) x Exposure(30%) x Vulnerability(30%)

#   School                                H      E      V   Score Risk
----------------------------------------------------------------------
#1  School Puducherry Border          93.0    85  19.9   68.7 CRITICAL
#2  Panchayat School Nagapattinam     65.8    85  55.0   68.3 CRITICAL
#3  School Ramanathapuram             44.4    85  79.2   67.0 CRITICAL
#4  School Kanchipuram                51.7    85  68.2   66.6 CRITICAL
#5  Panchayat School Tirunelveli      33.4    85  90.0   65.9 CRITICAL
#6  Govt School Cuddalore             58.9    85  55.0   65.6 CRITICAL
#7  School Villupuram                 47.1    85  59.8   62.3 CRITICAL
#8  School Tuticorin                  38.5    85  45.7   54.6 CRITICAL
#9  Govt High School Chennai          48.3    85   0.1   44.8 CRITICAL
#10 Govt School Tiruchirappalli       38.8    45  40.2   41.1 CRITICAL
#11 Govt School Thanjavur             39.2    45  15.0   33.

In [4]:
# Cell 7 — Save all outputs
from google.colab import files

# Save CSV
df_hev.to_csv('tamil_nadu_hev_risk.csv', index=False)
print('✅ HEV risk CSV saved!')

# Summary
print()
print('=== DAY 9 SUMMARY ===')
print(f"Hospitals found in Tamil Nadu: 2,168")
print(f"CRITICAL schools (H x E x V): {len(df_hev[df_hev.hev_risk == 'CRITICAL'])}")
print(f"HIGH schools: {len(df_hev[df_hev.hev_risk == 'HIGH'])}")
print(f"MEDIUM schools: {len(df_hev[df_hev.hev_risk == 'MEDIUM'])}")
print()
print('Most vulnerable school:',
      df_hev.nlargest(1, 'vulnerability_index').iloc[0]['school_name'])
print('Highest hazard school:',
      df_hev.nlargest(1, 'hazard_score').iloc[0]['school_name'])
print('Most isolated school:',
      df_hev.nlargest(1, 'nearest_hospital_km').iloc[0]['school_name'])

# Download
files.download('tamil_nadu_hev_risk.csv')
print()
print('✅ All files downloaded!')

✅ HEV risk CSV saved!

=== DAY 9 SUMMARY ===
Hospitals found in Tamil Nadu: 2,168
CRITICAL schools (H x E x V): 10
HIGH schools: 4
MEDIUM schools: 1

Most vulnerable school: Panchayat School Tirunelveli
Highest hazard school: School Puducherry Border
Most isolated school: Panchayat School Tirunelveli


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All files downloaded!
